# 🧬 RecapMM Voice Clone

**ဘာလုပ်ရမလဲ — အဆင့် ၃ ဆင့်ပဲ**

1. အောက်က cell ရဲ့ **▶** ကို တစ်ချက်နှိပ် (GPU ရွေးမေးရင် T4 / Connect)
2. ၅–၁၀ မိနစ်စောင့် — **အစိမ်းရောင် ကွက်** ပေါ်ရင် **📋 ကုဒ် ကူးမယ်** ကိုနှိပ်
3. RecapMM App ထဲ ပြန်သွား — ကုဒ်ကို အလိုအလျောက် ချိတ်ပေးမယ် (မချိတ်ရင် 📋 Paste နှိပ်)

⚠️ ဒီ page ကို **ဖွင့်ထားရမယ်** (ပိတ်ရင် Voice Clone ရပ်မယ်)။ နာရီအနည်းငယ်ကြာရင် Colab ပြတ်နိုင်တယ် — ပြတ်ရင် ▶ ပြန်နှိပ်ပြီး ကုဒ်အသစ် ထပ်ကူး။

GPU မရရင်: Runtime ▸ Change runtime type ▸ **T4 GPU**

In [ ]:
#@title ▶ ဒီ cell ကို run ပါ (ပြီးရင် ပေါ်လာတဲ့ ကုဒ်ကို App ထဲ ကူးထည့်) { display-mode: "form" }
import base64, os, re, subprocess, sys
from IPython.display import HTML, display

subprocess.run("nvidia-smi -L", shell=True)
print("⏳ ပထမဆုံးအကြိမ်မှာ install + model ဒေါင်းလို့ ၅–၁၀ မိနစ်လောက် ကြာမယ်…")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "voxcpm", "uvicorn[standard]", "starlette"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "resemblyzer"])      # optional: lets the app pick the take that sounds most like the reference

SRC = r'''"""RecapMM Voice Clone GPU worker - runs the VoxCPM2 model on a GPU box (Colab, RunPod, Vast.ai, a home PC ...).

It opens a public https tunnel (Cloudflare quick tunnel, no account needed) and tells the Recap server its address every ~45 s,
so the server sends every Voice Clone line here. Nothing on the phone changes.

Two ways to use it:
  * OWN (nothing to set): leave SERVER_URL empty. The worker makes its own secret key and prints a connection code (RCP1:...);
    paste that code into the app's Voice Clone page ("ကိုယ်ပိုင် GPU"). The Recap server is not involved.
  * SERVER: set SERVER_URL (e.g. https://recapmm.duckdns.org) and WORKER_KEY (= CLONE_WORKER_KEY in the server's .env);
    the worker then registers itself with the server and the server sends every line here.
"""
import base64
import hmac
import json
import os
import re
import secrets
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

import numpy as np
import uvicorn
from starlette.applications import Starlette
from starlette.concurrency import run_in_threadpool
from starlette.responses import JSONResponse, Response
from starlette.routing import Route

SERVER_URL = os.environ.get("SERVER_URL", "").strip().rstrip("/")          # empty = own mode
KEY = os.environ.get("WORKER_KEY", "").strip() or secrets.token_urlsafe(24)
PORT = int(os.environ.get("PORT", "8765"))
REPO = "openbmb/VoxCPM2"
REFS = "/tmp/recap_refs"
os.makedirs(REFS, exist_ok=True)
UA = {"User-Agent": "RecapMM-Worker/1.0", "Content-Type": "application/json"}

lock = threading.Lock()          # one synthesis at a time on one GPU
model = None
gpu_name = "CPU"


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def load_model():
    global model, gpu_name
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"
    log("GPU:", gpu_name)
    from voxcpm import VoxCPM
    try:
        model = VoxCPM.from_pretrained(REPO, load_denoiser=False)
    except TypeError:
        model = VoxCPM.from_pretrained(REPO)
    log("model loaded")
    load_scorer()


# ---- speaker-similarity scoring (optional: pip install resemblyzer) -> "best of N" picks the take that sounds most like the reference
_enc = None
_ref_emb = {}


def load_scorer():
    global _enc
    try:
        from resemblyzer import VoiceEncoder
        _enc = VoiceEncoder(device="cpu")
        log("similarity scorer ready")
    except Exception as e:  # noqa: BLE001
        log("similarity scorer not available (%s) - best-of-N is off" % str(e)[:80])


def _embed(x, sr):
    from resemblyzer import preprocess_wav
    return _enc.embed_utterance(preprocess_wav(x, source_sr=sr))


def ref_embedding(path):
    e = _ref_emb.get(path)
    if e is None:
        from resemblyzer import preprocess_wav
        e = _enc.embed_utterance(preprocess_wav(path))
        _ref_emb[path] = e
    return e


def gen_once(path, text, cfg, steps, prompt_text):
    kw = dict(text=text, reference_wav_path=path, cfg_value=cfg, inference_timesteps=steps)
    if prompt_text:      # "ultimate cloning": the clip + its exact transcript -> the model continues that very voice (much closer)
        kw.update(prompt_wav_path=path, prompt_text=prompt_text)
    with lock:
        try:
            wav = model.generate(retry_badcase=True, **kw)      # retries takes whose length is abnormal
        except TypeError:
            wav = model.generate(**kw)
        sr = int(model.tts_model.sample_rate)
    return np.asarray(wav, dtype=np.float32).reshape(-1), sr


def to_pcm(x, sr):
    if sr != 24000:                                   # the app expects 24 kHz mono 16-bit
        n = int(round(len(x) * 24000 / sr))
        if sr % 24000 == 0:
            k = sr // 24000
            x = x[: len(x) // k * k].reshape(-1, k).mean(axis=1)
        else:
            x = np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x)
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    if peak > 1.0:
        x = x / peak
    return (x * 32767).astype("<i2").tobytes()


def generate(path, text, cfg, steps, prompt_text="", best_of=1):
    """Returns (pcm bytes, similarity to the reference 0..1 or None)."""
    n = max(1, min(5, int(best_of))) if _enc is not None else 1
    if n == 1:
        x, sr = gen_once(path, text, cfg, steps, prompt_text)
        sim = None
        if _enc is not None:
            try:
                sim = float((_embed(x, sr) * ref_embedding(path)).sum())
            except Exception:  # noqa: BLE001
                sim = None
        return to_pcm(x, sr), sim
    best, best_sim = None, -1.0
    ref = ref_embedding(path)
    for _ in range(n):
        x, sr = gen_once(path, text, cfg, steps, prompt_text)
        sim = float((_embed(x, sr) * ref).sum())
        if sim > best_sim:
            best, best_sim = (x, sr), sim
        if sim >= 0.88:                               # close enough: stop early
            break
    return to_pcm(*best), best_sim


async def health(req):
    return JSONResponse({"ok": True, "gpu": gpu_name, "loaded": model is not None})


async def check(req):
    """The app's "connect" test: is the key right and is the model loaded?"""
    if not hmac.compare_digest(req.headers.get("x-key", ""), KEY):
        return JSONResponse({"ok": False}, 401)
    return JSONResponse({"ok": True, "gpu": gpu_name, "loaded": model is not None, "scorer": _enc is not None})


async def synth(req):
    if not hmac.compare_digest(req.headers.get("x-key", ""), KEY):
        return JSONResponse({"ok": False}, 401)
    if model is None:
        return JSONResponse({"ok": False, "error": "loading"}, 503)
    d = await req.json()
    rid = re.sub(r"[^A-Za-z0-9_-]", "", str(d.get("ref_id", "")))[:80]
    path = os.path.join(REFS, rid + ".wav")
    if d.get("ref") and rid:
        with open(path, "wb") as f:
            f.write(base64.b64decode(d["ref"]))
    if not rid or not os.path.isfile(path):
        return JSONResponse({"need_ref": True}, 409)          # the server then sends the reference audio
    text = str(d.get("text", ""))[:1500]
    try:
        steps = max(4, min(30, int(d.get("steps", 10))))
        cfg = max(1.0, min(4.0, float(d.get("cfg", 2.0))))
    except (TypeError, ValueError):
        steps, cfg = 10, 2.0
    prompt_text = str(d.get("prompt_text", "")).strip()[:600]
    try:
        best_of = max(1, min(5, int(d.get("best_of", 1))))
    except (TypeError, ValueError):
        best_of = 1
    try:
        pcm, sim = await run_in_threadpool(generate, path, text, cfg, steps, prompt_text, best_of)
    except Exception as e:  # noqa: BLE001
        log("synth error:", e)
        return JSONResponse({"ok": False, "error": str(e)[:200]}, 500)
    return Response(pcm, media_type="application/octet-stream", headers={"X-Sim": "%.3f" % sim} if sim is not None else None)


app = Starlette(routes=[Route("/health", health), Route("/check", check), Route("/synth", synth, methods=["POST"])])


# ------------------------------------------------------------------ tunnel + registration
def start_tunnel():
    exe = "/tmp/cloudflared"
    if not os.path.isfile(exe):
        log("downloading cloudflared ...")
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", exe)
        os.chmod(exe, 0o755)
    p = subprocess.Popen([exe, "tunnel", "--url", "http://127.0.0.1:%d" % PORT, "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    found = {}

    def reader():
        for line in p.stdout:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and "url" not in found:
                found["url"] = m.group(0)
    threading.Thread(target=reader, daemon=True).start()
    for _ in range(90):
        if "url" in found:
            return p, found["url"]
        time.sleep(1)
    p.kill()
    raise RuntimeError("tunnel did not start")


def register(url):
    req = urllib.request.Request(SERVER_URL + "/api/v1/internal/clone-worker", data=json.dumps({"url": url, "gpu": gpu_name}).encode(),
                                 headers={**UA, "X-Key": KEY}, method="POST")
    with urllib.request.urlopen(req, timeout=20) as r:
        return r.status == 200


def show_code(url):
    code = "RCP1:" + base64.urlsafe_b64encode(json.dumps({"u": url, "k": KEY, "g": gpu_name}).encode()).decode().rstrip("=")
    print("\n" + "=" * 60 + "\n📱 App › Voice Clone › ကိုယ်ပိုင် GPU မှာ ဒီကုဒ်ကို ကူးထည့်ပါ:\n\n" + code + "\n\n" + "=" * 60 + "\n", flush=True)


def supervisor():
    while model is None:
        time.sleep(1)
    while True:
        try:
            proc, url = start_tunnel()
            log("tunnel:", url)
            if not SERVER_URL:
                show_code(url)                        # own mode: the app talks to this worker directly
                while proc.poll() is None:
                    time.sleep(10)
                log("tunnel stopped, restarting ... (ကုဒ်အသစ် ထွက်လာမယ်)")
                continue
            ok_before = None
            while proc.poll() is None:
                try:
                    ok = register(url)
                    err = ""
                except urllib.error.HTTPError as e:
                    ok, err = False, "HTTP %d (CLONE_WORKER_KEY မကိုက်ဘူး / server မှာ မထည့်ရသေး)" % e.code
                except Exception as e:  # noqa: BLE001
                    ok, err = False, str(e)[:120]
                if ok != ok_before:
                    log("connected to Recap server ✅  — Voice Clone ကို App ကနေ သုံးလို့ရပါပြီ" if ok else "cannot register: " + err)
                    ok_before = ok
                time.sleep(45)
            log("tunnel stopped, restarting ...")
        except Exception as e:  # noqa: BLE001
            log("tunnel error:", e)
            time.sleep(10)


if __name__ == "__main__":
    threading.Thread(target=load_model, daemon=True).start()
    threading.Thread(target=supervisor, daemon=True).start()
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")
'''
open("clone_worker.py", "w", encoding="utf-8").write(SRC)

env = dict(os.environ, SERVER_URL="", WORKER_KEY="")      # own mode: no Recap server involved
p = subprocess.Popen([sys.executable, "-u", "clone_worker.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in p.stdout:                                      # keep this cell running - it IS the Voice Clone engine
    m = re.search(r"RCP1:[A-Za-z0-9_-]+", line)
    if m:
        c = m.group(0)
        display(HTML(
            '<div style="padding:14px;border-radius:12px;background:#0f3d2e;color:#fff;font-family:sans-serif">'
            '<div style="font-size:16px;margin-bottom:8px">✅ အသင့်ဖြစ်ပါပြီ — ကုဒ်ကို ကူးပြီး App ထဲ ပြန်သွားပါ</div>'
            '<input id="rcp" readonly value="' + c + '" style="width:100%;padding:8px;border-radius:8px;border:0;font-size:12px" onclick="this.select()">'
            '<button style="margin-top:10px;padding:10px 16px;border-radius:10px;border:0;font-size:16px" '
            'onclick="var i=document.getElementById(\'rcp\');i.select();document.execCommand(\'copy\');this.innerText=\'✔ ကူးပြီးပါပြီ\'">📋 ကုဒ် ကူးမယ်</button></div>'))
    else:
        print(line, end="")
